## 1- Post-processing of *GeoNames* and *MeSH* Knowledge Databases (KBs)

For GeoNames KB, we assume that you downloaded $\textit{allCountries}$ available at $\href{https://download.geonames.org/export/dump/allCountries.zip}{geonames.org}$. This version does not have any $\textit{headers}$. Those can be retrieved and mapped using the $\href{https://download.geonames.org/export/dump/readme.txt}{readme.txt}$ file.

For the MeSH KB, this dataset can be obtained by filling the form available at $\href{https://mesh.inserm.fr/telecharger-le-fichier-mesh/}{MeSH-Inserm}$

In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd

In [2]:
# GeoNames headers
headers = [
    "geonameid",
    "name",
    "asciiname",
    "alternatenames",
    "latitude",
    "longitude",
    "feature_class",
    "feature_code",
    "country_code",
    "cc2",
    "admin1_code",
    "admin2_code",
    "admin3_code",
    "admin4_code",
    "population",
    "elevation",
    "dem",
    "timezone",
    "modification_date",
]

df_geonames = pd.read_csv("../data/allCountries.txt", sep="\t", names=headers)
df_mesh = pd.read_json("../data/2026_js_elastic_mesh_bilingue.json", encoding="utf-8")

The previous lines of codes map the required headers for GeoNames and assume that the two KBs are inside the "*../data*" folder.

For MeSH KB, we are interested in the *trx* field which hold the the French Descriptor. The values of this field will be normalized (*i.e* Capitalized in our case) in order to simplify the indexation and search.

In [3]:
df_mesh = df_mesh[(df_mesh.db == "mesh")]
df_mesh["trx"] = df_mesh.trx.map(lambda x: x.lower().capitalize())

## 2- Entity Linking

### 2.1- LOCATION

The following code is implementing A Zero-Shot Location Linker:

In [4]:
import os
import json
import re
from typing import List, Dict, Any

from ollama import chat
from ollama import ChatResponse


class ZeroShotGeoLinker:

    def __init__(self):

        self.df = df_geonames
        self.df['population'] = self.df['population'].fillna(0).astype(int)
        self.df['name_lower'] = self._normalize(self.df['name'].str.lower())

    def _normalize(self, name: str) -> str:
        for ee in ["é", "è", "ê"]:
            name = name.replace(ee, "e")
        for aa in ["à", "â"]:
            name = name.replace(aa, "a")
        return name

    def get_candidates(self, mention: str, limit: int = 100) -> List[Dict[str, Any]]:
        """
        Retrieve candidates
        """
        mention_lower = mention.lower()
        # Exact match
        exact = self.df[self.df['name_lower'] == mention_lower].sort_values("population", ascending=False)
        if len(exact) >= 3:
            candidates_df = exact.head(limit)
        else:
            # Partial match (contains)
            partial = self.df[self.df['name_lower'].str.contains(mention_lower, na=False, regex=False)]
            candidates_df = pd.concat([exact, partial]).drop_duplicates(subset='geonameid').head(limit)
        return [self._row_to_dict(row) for row in candidates_df.to_dict('records')]

    def _row_to_dict(self, row: pd.Series) -> Dict[str, Any]:
        """Convert a DataFrame row to a dictionary"""
        return {
            'geonameid': int(row['geonameid']),
            'name': str(row['name']),
            'country_code': str(row['country_code']) if pd.notna(row['country_code']) else None,
            'admin1_code': str(row['admin1_code']) if pd.notna(row['admin1_code']) else None,
            'latitude': float(row['latitude']) if pd.notna(row['latitude']) else None,
            'longitude': float(row['longitude']) if pd.notna(row['longitude']) else None,
            'population': int(row['population']) if pd.notna(row['population']) else 0,
            'feature_code': str(row['feature_code']) if pd.notna(row['feature_code']) else None
        }

    def format_candidate_description(self, candidate: Dict[str, Any]) -> str:
        """Create a human‑readable description for a candidate"""
        desc = candidate['name']
        if candidate['country_code']:
            desc += f" ({candidate['country_code']}"
            if candidate['admin1_code']:
                desc += f", {candidate['admin1_code']}"
            desc += ")"
        feature_map = {
            'PPLC': 'capital city', 'PPL': 'city', 'PPLA2': 'seat of a second-order admin division',
            'ADM1': 'first-order administrative division', 'RIV': 'river', 'MT': 'mountain',
            'PPLA': 'seat of a first-order admin division', 'PPLA1': 'capital of a country'
        }
        fcode = candidate.get('feature_code')
        if fcode and fcode in feature_map:
            desc += f" – {feature_map[fcode]}"
        if candidate['population'] > 0:
            desc += f" (population {candidate['population']:,})"
        return desc

    def build_prompt(self, context: str, mention: str, candidates: List[Dict[str, Any]]) -> str:
        """Build the zero‑shot prompt for the LLM."""
        if not candidates:
            return f"Context: {context}\n\nNo candidate locations found for '{mention}'. Answer: NONE"

        lines = [f"Context: {context}\n"]
        lines.append(f"Which of the following locations does '{mention}' refer to?\n")
        for idx, cand in enumerate(candidates, start=1):
            desc = self.format_candidate_description(cand)
            lines.append(f"{idx}. {desc}")
        lines.append("\nAnswer with the number only (1, 2, 3, 4, 5, ...) or 0 if none of the above.")
        return "\n".join(lines)
    
    def extract_context_by_offsets(self, document: str, start_char: int, end_char: int, 
                                   window_chars: int = 150, trim_to_word: bool = True) -> str:
        """
        Extract a window of text around a mention using exact character offsets
        """
        doc_len = len(document)
        left = max(0, start_char - window_chars)
        right = min(doc_len, end_char + window_chars)

        if trim_to_word:
            if left > 0:
                while left < start_char and document[left] != ' ':
                    left += 1
                if left < doc_len and document[left] == ' ':
                    left += 1
            if right < doc_len:
                while right > end_char and document[right - 1] != ' ':
                    right -= 1

        context = document[left:right].strip()
        return context

    def disambiguate_with_offsets(self, document: str, mention: str, 
                                  start_char: int, end_char: int,
                                  window_chars: int = 200,
                                  model: str = "mistral") -> Dict[str, Any]:
        """
        Disambiguate a mention using exact character offsets to extract context
        """
        context = self.extract_context_by_offsets(document, start_char, end_char, window_chars)
        return self.disambiguate(context, mention, model)

    def disambiguate(self, context: str, mention: str, model: str = "mistral") -> Dict[str, Any]:
        """
        Main method: given a mention and its context, return the best GeoName ID
        """
        candidates = self.get_candidates(mention)
        if not candidates:
            return {
                "mention": mention,
                "selected_geonameid": None,
            }

        prompt = self.build_prompt(context, mention, candidates)

        try:
            response = chat(
                model=model,
                messages=[{"role": "user", "content": prompt}],
            )
            answer = response.message.content.strip()
        except Exception as e:
            return {
                "mention": mention,
                "selected_geonameid": None,
                "error": f"LLM API error: {str(e)}"
            }

        match = re.search(r'\d+', answer)
        if not match:
            return {
                "mention": mention,
                "selected_geonameid": None,
            }
        choice = int(match.group())

        if 1 <= choice <= len(candidates):
            selected = candidates[choice-1]
            return {
                "mention": mention,
                "selected_geonameid": selected['geonameid'],
            }
        else:
            return {
                "mention": mention,
                "selected_geonameid": None,
            }


### 2.2- Indexing the French Descriptor *trx*

In [5]:
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.utils import embedding_functions
from chromadb.config import Settings
from tqdm import tqdm

In [6]:
chroma_client = chromadb.PersistentClient(path="../data/chroma_db_mesh")
collection = chroma_client.get_or_create_collection(name="rag_mesh")


In [ ]:
embedder = SentenceTransformer("dangvantuan/sentence-camembert-base")

In [ ]:
for index, row in tqdm(df_mesh.iterrows(), total=df_mesh.shape[0]):
    name = row.trx
    Id = str(row.id)

    if name:
        embeddings = embedder.encode(name).tolist()
        
        collection.add(
            embeddings=embeddings,
            documents=name,
            ids=Id
        )

### 2.3- Runs

In [8]:
df_train = pd.read_json("../data/train_evalLLM.json", encoding="utf-8")
geo_linker = ZeroShotGeoLinker()

#### 2.3.1- Run1

In [ ]:
linked_entities = []
window = 150

for _, row in tqdm(df_train.iterrows(), total=df_train.shape[0]):
    text = row.text
    entities = row.entities
    ents = []

    for entity in entities:
        ent = {
            "start": entity["start"],
            "end": entity["end"],
            "id": entity["id"],
            "text": entity["text"],
            "id_kb": "",
            "source": "",
            "label": entity["label"]
        }
        
        if entity["label"] == "LOCATION":
            location = entity["text"]
            selected_location = geo_linker.disambiguate_with_offsets(text, location, entity["start"][0], entity["end"][0])
            Id = selected_location["selected_geonameid"]
            if Id:
                ent["id_kb"] = str(Id)
                ent["source"] = "GeoNames"
        
        else:
            rows = df_mesh[(df_mesh.trx == entity["text"].capitalize())]
            if not rows.empty:
                ent["id_kb"] = rows.iloc[0].id
                ent["source"] = "MeSH"
            else:
                query = geo_linker.extract_context_by_offsets(text, entity["start"][0], entity["end"][0], window)
                query_embedding = embedder.encode([query]).tolist()[0]

                results = collection.query(
                    query_embeddings=[query_embedding],
                    n_results=1,
                    include=["documents", "distances"]
                )
                
                ent["id_kb"] = results["ids"][0][0]
                ent["source"] = "MeSH"

        
        ents.append(ent)
            
            
    linked_entities.append(
        {
            "text": text,
            "entities": ents
        }
    )